# 02 — Préparation des données

**Objectif :** définir deux populations d'analyse et préparer une matrice client-produit, sans modifier les données originales. Les calculs réutilisables se trouvent dans `src/`.

## 1. Import des fonctions et chargement

Le CSV original doit être placé dans `data/données.csv`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Fonctionne que Jupyter démarre dans la racine ou dans notebooks/
racine = Path.cwd()
if not (racine / 'src').is_dir():
    racine = racine.parent
if not (racine / 'src').is_dir():
    raise FileNotFoundError('Ouvrir le notebook depuis le projet ou notebooks/.')
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

from src.chargement import charger_donnees, colonnes_produits
from src.preparation import (
    verifier_produits, decouper_a_la_premiere_rupture,
    base_initiale, base_agregee,
    frequences_produits, comparer_frequences,
    base_identifiants_distincts,
)

df = charger_donnees(racine)
produits = colonnes_produits(df)
verifier_produits(df, produits)
print(f'Données : {df.shape[0]} lignes, {len(produits)} produits')

Données : 73930 lignes, 26 produits


## 2. Définition des populations

**Hypothèse A (principale)** : retenir le segment initial, qui comporte des identifiants uniques. **Hypothèse B (complémentaire)** : agréger toutes les lignes d'un même identifiant avec le maximum des indicateurs binaires.

Ces hypothèses sont testées, pas présentées comme des certitudes sur l'origine des données.

In [2]:
# Découpage des données
segment_initial, segment_final, limite = decouper_a_la_premiere_rupture(df)

# Construction des trois matrices
matrice_initiale = base_initiale(df, produits)
matrice_distincte = base_identifiants_distincts(df, produits)
matrice_agregee = base_agregee(df, produits)

# Comparaison des scénarios
bilan = pd.DataFrame({
    'scenario': [
        'Segment initial',
        'Identifiants distincts',
        'Données complètes agrégées'
    ],
    'nombre_clients': [
        len(matrice_initiale),
        len(matrice_distincte),
        len(matrice_agregee)
    ],
    'nombre_produits': [
        len(matrice_initiale.columns),
        len(matrice_distincte.columns),
        len(matrice_agregee.columns)
    ],
})

display(bilan)
display(matrice_distincte.head())

,scenario,nombre_clients,nombre_produits
0,Segment initial,72531,26
1,Identifiants distincts,72638,26
2,Données complètes agrégées,72638,26


,CART,CJEU,PACKSERV,DAV,DECAUTH,CPAI,CARTE,auto,GAV,PJ,...,ASSMH,CREDHAB,CSL,OBLIGATION,SICAVFCP,EPARBANC,ASSPREVDEC,ASSDEP,CREDRENOV,ASSPREVDECDEP
id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,1,0,0,0,1,0,0,...,1,0,1,0,0,1,0,0,0,0
2,1,0,1,1,0,1,0,1,1,1,...,1,0,1,0,1,1,0,0,0,0
3,1,1,1,1,1,1,0,0,1,1,...,1,0,1,0,0,0,0,0,0,0
4,1,0,1,1,1,1,0,0,1,0,...,1,0,1,0,0,0,0,0,0,0
5,1,0,0,1,0,1,0,0,1,0,...,1,0,1,0,0,0,0,0,0,0


## 3. Fréquence de détention des produits

Chaque pourcentage correspond à la part des clients de la population retenue qui détiennent le produit.

In [3]:
display(frequences_produits(matrice_initiale).round(2))

,pourcentage_clients
DAV,99.04
CART,84.98
CSL,84.25
PACKSERV,82.77
CPAI,79.81
ASSMH,64.23
DECAUTH,64.09
CREDCONSO,44.29
PJ,42.44
EPARLOG,36.94


## 4. Comparaison descriptive des scénarios

On compare les fréquences de chaque produit entre les deux approches pour voir si notre choix de population change les résultats. L'écart est exprimé en **points de pourcentage**.

In [3]:
# Fréquence de détention des produits (%)
frequences = pd.DataFrame({
    'Segment initial': matrice_initiale.mean() * 100,
    'Identifiants distincts': matrice_distincte.mean() * 100,
    'Données complètes agrégées': matrice_agregee.mean() * 100
})

# Écarts par rapport à la base initiale
frequences['Écart distincts (pp)'] = (
    frequences['Identifiants distincts']
    - frequences['Segment initial']
)

frequences['Écart agrégée (pp)'] = (
    frequences['Données complètes agrégées']
    - frequences['Segment initial']
)

display(frequences.round(3))

,Segment initial,Identifiants distincts,Données complètes agrégées,Écart distincts (pp),Écart agrégée (pp)
CART,84.982,85.004,85.008,0.022,0.026
CJEU,11.067,11.051,11.051,-0.016,-0.016
PACKSERV,82.766,82.791,82.798,0.025,0.032
DAV,99.039,99.040,99.042,0.001,0.003
DECAUTH,64.087,64.140,64.155,0.053,0.068
CPAI,79.813,79.843,79.852,0.030,0.039
CARTE,1.591,1.736,1.753,0.145,0.161
auto,32.943,33.042,33.054,0.099,0.111
GAV,10.466,10.598,10.614,0.132,0.148
PJ,42.437,42.522,42.536,0.085,0.099


## 5. Conclusion de la préparation des données

La préparation des données a permis de construire trois matrices client-produit selon différentes méthodes de traitement des identifiants répétés.

Le segment initial contient **72 531 observations**, tandis que les deux autres matrices comportent **72 638 identifiants distincts**, soit une différence de 107 identifiants.

La comparaison des fréquences de détention des produits montre que les écarts entre les trois scénarios restent globalement faibles, avec une différence maximale de 0,161 point de pourcentage pour le produit CARTE.

Nous avons donc choisi de **retenir le segment initial comme base principale pour la suite de l'analyse**, car il présente des identifiants uniques et ne nécessite aucune agrégation supplémentaire. Ce choix permet également d'éviter de fusionner des observations dont la signification exacte reste inconnue.

Toutefois, nous ne pouvons pas affirmer que les observations du segment final sont erronées. Ce choix constitue donc une hypothèse de travail, fondée sur les particularités observées lors de l'audit des données.

La base complète agrégée sera conservée comme référence complémentaire afin de vérifier si le choix de la population influence les associations entre produits.

### Prochaine étape

Nous allons construire la matrice d'association et calculer les indicateurs de **support, confiance, confiance attendue et lift**, conformément à la méthodologie du cours.

L'analyse portera particulièrement sur les produits d'assurance et d'épargne, afin d'identifier des associations pertinentes pour proposer des actions de cross-selling.